<a href="https://colab.research.google.com/github/iamprakhar10/Fine-tuning/blob/main/qlora_banking77_part1_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QLoRA Fine-Tuning Project — Banking77

This notebook is the clean **start-to-baseline** version of the project.

## Goal

We will:

1. Load the current Parquet-backed `mteb/banking77` dataset.
2. Keep 10 banking intents so the first experiment stays small.
3. Convert examples into conversational prompt/completion format for SFT.
4. Load `Qwen/Qwen3-1.7B` in 4-bit NF4.
5. Run deterministic baseline inference **before fine-tuning**.
6. Evaluate and save baseline results.

The next notebook stage will attach LoRA adapters and perform SFT.

> In Google Colab, select **Runtime → Change runtime type → GPU** before running the notebook.


## 1. Install dependencies

In [1]:
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


If Colab asks you to restart the runtime after installation, restart it and continue from the imports cell below.

## 2. Imports and environment check

In [2]:
import json
from pathlib import Path

import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb

from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("bitsandbytes:", bnb.__version__)

PyTorch: 2.11.0+cu130
Transformers: 5.18.0
Datasets: 5.1.0
PEFT: 0.21.2
TRL: 1.14.1
bitsandbytes: 0.50.2


## 3. Verify that Colab has a GPU

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU was detected. In Colab select "
        "Runtime → Change runtime type → GPU, then rerun the notebook."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version reported by PyTorch:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())

GPU: Tesla T4
CUDA version reported by PyTorch: 13.0
BF16 supported: True


## 4. Project constants

In [4]:
MODEL_ID = "Qwen/Qwen3-1.7B"
DATASET_ID = "mteb/banking77"

SEED = 42
BASELINE_EVAL_SIZE = 100

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_ID)
print("Dataset:", DATASET_ID)

Model: Qwen/Qwen3-1.7B
Dataset: mteb/banking77


## 5. Load Banking77

We use `mteb/banking77`, which is stored as Parquet and does not rely on the old Python dataset script that caused the earlier error.

In [5]:
dataset = load_dataset(DATASET_ID)

print(dataset)
print("\nFirst training example:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/16.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

First training example:
{'text': 'I am still waiting on my card?', 'label': 11, 'label_text': 'card_arrival'}


## 6. Inspect labels

In [6]:
all_labels = sorted(set(dataset["train"]["label_text"]))

print("Number of unique labels:", len(all_labels))
print("\nFirst 20 labels:")
for label in all_labels[:20]:
    print("-", label)

Number of unique labels: 77

First 20 labels:
- Refund_not_showing_up
- activate_my_card
- age_limit
- apple_pay_or_google_pay
- atm_support
- automatic_top_up
- balance_not_updated_after_bank_transfer
- balance_not_updated_after_cheque_or_cash_deposit
- beneficiary_not_allowed
- cancel_transfer
- card_about_to_expire
- card_acceptance
- card_arrival
- card_delivery_estimate
- card_linking
- card_not_working
- card_payment_fee_charged
- card_payment_not_recognised
- card_payment_wrong_exchange_rate
- card_swallowed


## 7. Select 10 intents

We intentionally start with 10 classes. The goal of this first project is to learn the fine-tuning pipeline, not to maximize benchmark difficulty.

In [7]:
CHOSEN_LABELS = [
    "activate_my_card",
    "age_limit",
    "atm_support",
    "cancel_transfer",
    "card_arrival",
    "card_not_working",
    "cash_withdrawal_charge",
    "change_pin",
    "compromised_card",
    "cash_withdrawal_not_recognised",
]

missing_labels = sorted(set(CHOSEN_LABELS) - set(all_labels))

if missing_labels:
    raise ValueError(
        f"These requested labels were not found in the dataset: {missing_labels}"
    )

train_ds = dataset["train"].filter(
    lambda example: example["label_text"] in CHOSEN_LABELS
)

test_ds = dataset["test"].filter(
    lambda example: example["label_text"] in CHOSEN_LABELS
)

print("Filtered training examples:", len(train_ds))
print("Filtered test examples:", len(test_ds))

Filter:   0%|          | 0/9993 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3076 [00:00<?, ? examples/s]

Filtered training examples: 1319
Filtered test examples: 399


## 8. Create the instruction prompt

In [8]:
allowed_labels = "\n".join(f"- {label}" for label in CHOSEN_LABELS)

SYSTEM_PROMPT = f"""You are a banking support intent classifier.

Classify the user's request into exactly one of the following labels:

{allowed_labels}

Return ONLY the label. Do not add explanations, punctuation, or extra text."""

print(SYSTEM_PROMPT)

You are a banking support intent classifier.

Classify the user's request into exactly one of the following labels:

- activate_my_card
- age_limit
- atm_support
- cancel_transfer
- card_arrival
- card_not_working
- cash_withdrawal_charge
- change_pin
- compromised_card
- cash_withdrawal_not_recognised

Return ONLY the label. Do not add explanations, punctuation, or extra text.


## 9. Convert dataset examples to SFT prompt/completion format

In [9]:
def to_sft_example(example: dict) -> dict:
    """
    Convert one Banking77 example into conversational prompt/completion format.

    The prompt contains the system instruction and the customer's banking
    query. The completion contains the ground-truth intent label that the
    assistant should generate during supervised fine-tuning.

    Parameters
    ----------
    example : dict
        A Banking77 example containing at least the fields ``text`` and
        ``label_text``.

    Returns
    -------
    dict
        A dictionary with two fields:

        - ``prompt``: system and user chat messages.
        - ``completion``: the expected assistant message containing the
          correct intent label.
    """
    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": example["text"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["label_text"],
            }
        ],
    }


train_sft = train_ds.map(
    to_sft_example,
    remove_columns=train_ds.column_names,
)

test_sft = test_ds.map(
    to_sft_example,
    remove_columns=test_ds.column_names,
)

print(train_sft[0])

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/399 [00:00<?, ? examples/s]

{'prompt': [{'role': 'system', 'content': "You are a banking support intent classifier.\n\nClassify the user's request into exactly one of the following labels:\n\n- activate_my_card\n- age_limit\n- atm_support\n- cancel_transfer\n- card_arrival\n- card_not_working\n- cash_withdrawal_charge\n- change_pin\n- compromised_card\n- cash_withdrawal_not_recognised\n\nReturn ONLY the label. Do not add explanations, punctuation, or extra text."}, {'role': 'user', 'content': 'I am still waiting on my card?'}], 'completion': [{'role': 'assistant', 'content': 'card_arrival'}]}


At this point we have the format that will later be passed to TRL's `SFTTrainer`. We are **not training yet**.

## 10. Choose the 4-bit compute dtype

In [10]:
COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("4-bit compute dtype:", COMPUTE_DTYPE)

4-bit compute dtype: torch.bfloat16


## 11. Load the tokenizer

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("EOS token:", tokenizer.eos_token)
print("EOS token id:", tokenizer.eos_token_id)
print("Pad token id:", tokenizer.pad_token_id)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

EOS token: <|im_end|>
EOS token id: 151645
Pad token id: 151643


## 12. Configure 4-bit NF4 quantization

The base model is loaded in 4-bit to reduce memory usage. Later, QLoRA will keep these base weights frozen and train LoRA adapters.

In [12]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

print(quantization_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## 13. Load the base model

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype="auto",
)

model.eval()

print("Model loaded.")
print("Primary device:", model.device)

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.
Primary device: cuda:0


## 14. Define deterministic baseline inference

Qwen3 supports a thinking mode. For this short classification task we explicitly disable it.

We also use greedy decoding (`do_sample=False`) so that the benchmark is deterministic.

In [14]:
@torch.inference_mode()
def predict_intent(query: str) -> str:
    """
    Predict a Banking77 intent label for one customer query.

    The function formats the query using Qwen3's chat template, explicitly
    disables thinking mode, performs deterministic greedy generation, and
    returns only the newly generated text.

    Parameters
    ----------
    query : str
        A banking-related customer request to classify.

    Returns
    -------
    str
        The raw intent text generated by the current model.
    """
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": query,
        },
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return prediction.strip()

## 15. Sanity-check a few manual queries

In [15]:
manual_queries = [
    "My card still hasn't arrived.",
    "How do I change my PIN?",
    "Someone got access to my card details.",
    "Why was I charged for withdrawing cash?",
    "I want to cancel a transfer.",
]

for query in manual_queries:
    print("Query:     ", query)
    print("Prediction:", predict_intent(query))
    print("-" * 70)

Query:      My card still hasn't arrived.
Prediction: card_not_working
----------------------------------------------------------------------
Query:      How do I change my PIN?
Prediction: change_pin
----------------------------------------------------------------------
Query:      Someone got access to my card details.
Prediction: compromised_card
----------------------------------------------------------------------
Query:      Why was I charged for withdrawing cash?
Prediction: cash_withdrawal_charge
----------------------------------------------------------------------
Query:      I want to cancel a transfer.
Prediction: cancel_transfer
----------------------------------------------------------------------


## 16. Create a fixed baseline evaluation subset

In [16]:
baseline_eval_ds = test_ds.shuffle(seed=SEED).select(
    range(min(BASELINE_EVAL_SIZE, len(test_ds)))
)

print("Baseline evaluation examples:", len(baseline_eval_ds))
print(baseline_eval_ds[0])

Baseline evaluation examples: 100
{'text': "There is an unauthorized cash withdrawal from my account for 500£.  I definitely didn't make this withdrawal.  I need your help.", 'label': 20, 'label_text': 'cash_withdrawal_not_recognised'}


## 17. Label normalization helper

In [17]:
def normalize_label(text: str) -> str:
    """
    Normalize a model-generated label for exact-match evaluation.

    The function lowercases the output, keeps only the first non-empty line,
    and removes harmless surrounding whitespace and quoting characters. It
    does not try to infer a label from a long explanation because the model
    was explicitly instructed to return only one label.

    Parameters
    ----------
    text : str
        Raw generated model output.

    Returns
    -------
    str
        Normalized label text used for exact-match comparison.
    """
    normalized = text.strip().lower()

    non_empty_lines = [
        line.strip()
        for line in normalized.splitlines()
        if line.strip()
    ]

    if not non_empty_lines:
        return ""

    normalized = non_empty_lines[0]
    normalized = normalized.strip("`'\" ")

    return normalized

## 18. Baseline evaluation function

In [18]:
def evaluate_model(eval_dataset) -> dict:
    """
    Evaluate the current model using exact-match intent classification accuracy.

    Each query is sent through ``predict_intent``. The normalized generated
    label is compared with the normalized ground-truth ``label_text`` value.

    Parameters
    ----------
    eval_dataset : datasets.Dataset
        A dataset containing the columns ``text`` and ``label_text``.

    Returns
    -------
    dict
        Evaluation results containing:

        - ``accuracy``: fraction of predictions that exactly match the target.
        - ``correct``: number of correct predictions.
        - ``total``: number of evaluated examples.
        - ``predictions``: per-example query, target, prediction, and result.
    """
    predictions = []
    correct = 0

    for example in tqdm(eval_dataset):
        query = example["text"]
        expected = normalize_label(example["label_text"])
        predicted = normalize_label(predict_intent(query))

        is_correct = predicted == expected
        correct += int(is_correct)

        predictions.append(
            {
                "query": query,
                "expected": expected,
                "predicted": predicted,
                "correct": is_correct,
            }
        )

    total = len(eval_dataset)
    accuracy = correct / total if total else 0.0

    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "predictions": predictions,
    }

## 19. Run the baseline evaluation

In [19]:
baseline_results = evaluate_model(baseline_eval_ds)

print(
    f"Baseline accuracy: "
    f"{baseline_results['correct']}/{baseline_results['total']} "
    f"= {baseline_results['accuracy']:.2%}"
)

  0%|          | 0/100 [00:00<?, ?it/s]

Baseline accuracy: 79/100 = 79.00%


## 20. Inspect baseline mistakes

In [20]:
baseline_mistakes = [
    result
    for result in baseline_results["predictions"]
    if not result["correct"]
]

print("Number of mistakes:", len(baseline_mistakes))

for mistake in baseline_mistakes[:10]:
    print("Query:    ", mistake["query"])
    print("Expected: ", mistake["expected"])
    print("Predicted:", mistake["predicted"])
    print("-" * 70)

Number of mistakes: 21
Query:     My card broke.
Expected:  card_not_working
Predicted: compromised_card
----------------------------------------------------------------------
Query:     I thought cash withdrawals were free? Why have I been charged a fee? Also, how much is this fee?
Expected:  cash_withdrawal_charge
Predicted: cash_withdrawal_not_recognised
----------------------------------------------------------------------
Query:     I accidentally made a transaction to the wrong account.
Expected:  cancel_transfer
Predicted: cash_withdrawal_not_recognised
----------------------------------------------------------------------
Query:     I'm still waiting on my card to be delivered.
Expected:  card_arrival
Predicted: card_not_working
----------------------------------------------------------------------
Query:     I want to go back on what I did this morning
Expected:  cancel_transfer
Predicted: compromised_card
----------------------------------------------------------------------


## 21. Save baseline results

In [21]:
baseline_path = RESULTS_DIR / "baseline.json"

with baseline_path.open("w", encoding="utf-8") as file:
    json.dump(
        baseline_results,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Saved:", baseline_path.resolve())

Saved: /content/results/baseline.json


## 22. Stop here

At this point you should have:

- a working dataset,
- a 10-class SFT dataset,
- a 4-bit Qwen3-1.7B base model,
- deterministic inference,
- a fixed baseline evaluation set,
- `results/baseline.json`.

**Do not attach LoRA until this notebook runs cleanly through this point.**

The next stage will:

1. prepare the 4-bit model for k-bit training,
2. attach LoRA adapters,
3. inspect trainable parameter counts,
4. configure `SFTTrainer`,
5. train,
6. evaluate on the exact same baseline subset,
7. compare before vs. after.

### Current references

- Qwen3-1.7B model card: https://huggingface.co/Qwen/Qwen3-1.7B
- Banking77 Parquet dataset: https://huggingface.co/datasets/mteb/banking77
- Transformers bitsandbytes guide: https://huggingface.co/docs/transformers/main/quantization/bitsandbytes
- TRL SFTTrainer: https://huggingface.co/docs/trl/main/sft_trainer


In [22]:
from peft import prepare_model_for_kbit_training

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

print("4-bit base model prepared for training.")

4-bit base model prepared for training.


In [23]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules='all-linear', exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


In [24]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config,
)

model.print_trainable_parameters()

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


In [25]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_params / total_params:.4f}%"
)

Trainable parameters: 17,432,576
Total parameters:     1,033,364,480
Trainable percentage: 1.6870%


In [26]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="outputs/qlora-banking77",

    # Training duration
    num_train_epochs=2,

    # Small batches because we are on Colab
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # LoRA adapters can generally use a larger LR
    learning_rate=2e-4,

    # Sequence configuration
    max_length=256,
    packing=False,

    # We only want loss on the assistant completion
    completion_only_loss=True,

    # Memory optimization
    gradient_checkpointing=True,

    # Precision
    bf16=(COMPUTE_DTYPE == torch.bfloat16),
    fp16=(COMPUTE_DTYPE == torch.float16),

    # Logging
    logging_steps=10,
    logging_first_step=True,
    report_to="none",

    # We will save the adapter ourselves afterward
    save_strategy="no",

    # Reproducibility
    seed=SEED,
)

In [27]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    processing_class=tokenizer,
)

print("Trainer ready.")

Tokenizing train dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1319 [00:00<?, ? examples/s]

Trainer ready.


In [29]:
train_result = trainer.train()

Step,Training Loss
1,2.349675
10,0.276458
20,0.062846
30,0.026710
40,0.023219
50,0.012934
60,0.028456
70,0.013350
80,0.021397
90,0.003563


In [30]:
print(train_result.metrics)
trainer.state.log_history[-10:]

{'train_runtime': 1996.8048, 'train_samples_per_second': 1.321, 'train_steps_per_second': 0.083, 'total_flos': 3051425659889664.0, 'train_loss': 0.04268691321015807, 'entropy': 0.004359157488744875, 'num_tokens': 332762.0, 'mean_token_accuracy': 1.0, 'epoch': 2.0}


[{'loss': 0.021396981179714204,
  'grad_norm': 0.3203125,
  'learning_rate': 0.00010481927710843373,
  'entropy': 0.01280968616483733,
  'num_tokens': 161506.0,
  'mean_token_accuracy': 0.9936997994780541,
  'epoch': 0.9696969696969697,
  'step': 80},
 {'loss': 0.0035628747195005418,
  'grad_norm': 0.166015625,
  'learning_rate': 9.27710843373494e-05,
  'entropy': 0.010641763427932011,
  'num_tokens': 180506.0,
  'mean_token_accuracy': 0.999269005499388,
  'epoch': 1.084848484848485,
  'step': 90},
 {'loss': 0.011226190626621247,
  'grad_norm': 0.028076171875,
  'learning_rate': 8.072289156626507e-05,
  'entropy': 0.007634519894781988,
  'num_tokens': 200614.0,
  'mean_token_accuracy': 0.996870057284832,
  'epoch': 1.206060606060606,
  'step': 100},
 {'loss': 0.003439059481024742,
  'grad_norm': 0.0019073486328125,
  'learning_rate': 6.867469879518072e-05,
  'entropy': 0.007225406524958089,
  'num_tokens': 220903.0,
  'mean_token_accuracy': 0.9987986266613007,
  'epoch': 1.327272727272

In [33]:
ADAPTER_DIR = Path("adapter/banking77-qwen3-1.7b-qlora")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

Adapter saved to: adapter/banking77-qwen3-1.7b-qlora


In [34]:
!du -sh adapter/banking77-qwen3-1.7b-qlora

45M	adapter/banking77-qwen3-1.7b-qlora


In [35]:
model.eval()
model.config.use_cache = True

print("Model switched to evaluation mode.")

Model switched to evaluation mode.


In [36]:
manual_queries = [
    "My card still hasn't arrived.",
    "How do I change my PIN?",
    "Someone got access to my card details.",
    "Why was I charged for withdrawing cash?",
    "I want to cancel a transfer.",
]

for query in manual_queries:
    print("Query:     ", query)
    print("Prediction:", predict_intent(query))
    print("-" * 70)

Query:      My card still hasn't arrived.
Prediction: card_arrival
----------------------------------------------------------------------
Query:      How do I change my PIN?
Prediction: change_pin
----------------------------------------------------------------------
Query:      Someone got access to my card details.
Prediction: compromised_card
----------------------------------------------------------------------
Query:      Why was I charged for withdrawing cash?
Prediction: cash_withdrawal_charge
----------------------------------------------------------------------
Query:      I want to cancel a transfer.
Prediction: cancel_transfer
----------------------------------------------------------------------


In [37]:
test_ds.shuffle(seed=42)

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 399
})

In [38]:
finetuned_results = evaluate_model(
    baseline_eval_ds
)

  0%|          | 0/100 [00:00<?, ?it/s]

In [39]:
print(
    f"Fine-tuned accuracy: "
    f"{finetuned_results['correct']}/"
    f"{finetuned_results['total']} "
    f"= {finetuned_results['accuracy']:.2%}"
)

Fine-tuned accuracy: 100/100 = 100.00%


In [40]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [41]:
print(
    f"Base model accuracy:       "
    f"{baseline_results['accuracy']:.2%}"
)

print(
    f"QLoRA model accuracy:      "
    f"{finetuned_results['accuracy']:.2%}"
)

improvement = (
    finetuned_results["accuracy"]
    - baseline_results["accuracy"]
)

print(
    f"Absolute improvement:      "
    f"{improvement:+.2%}"
)

Base model accuracy:       79.00%
QLoRA model accuracy:      100.00%
Absolute improvement:      +21.00%


In [42]:
finetuned_path = RESULTS_DIR / "finetuned.json"

with finetuned_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        finetuned_results,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Saved: {finetuned_path}")

Saved: results/finetuned.json


In [43]:
improved_examples = []

for before, after in zip(
    baseline_results["predictions"],
    finetuned_results["predictions"],
):
    if (
        not before["correct"]
        and after["correct"]
    ):
        improved_examples.append(
            {
                "query": before["query"],
                "expected": before["expected"],
                "before": before["predicted"],
                "after": after["predicted"],
            }
        )

print(
    f"Examples fixed by fine-tuning: "
    f"{len(improved_examples)}"
)

Examples fixed by fine-tuning: 21


In [44]:
for example in improved_examples[:10]:
    print("Query:   ", example["query"])
    print("Expected:", example["expected"])
    print("Before:  ", example["before"])
    print("After:   ", example["after"])
    print("-" * 70)

Query:    My card broke.
Expected: card_not_working
Before:   compromised_card
After:    card_not_working
----------------------------------------------------------------------
Query:    I thought cash withdrawals were free? Why have I been charged a fee? Also, how much is this fee?
Expected: cash_withdrawal_charge
Before:   cash_withdrawal_not_recognised
After:    cash_withdrawal_charge
----------------------------------------------------------------------
Query:    I accidentally made a transaction to the wrong account.
Expected: cancel_transfer
Before:   cash_withdrawal_not_recognised
After:    cancel_transfer
----------------------------------------------------------------------
Query:    I'm still waiting on my card to be delivered.
Expected: card_arrival
Before:   card_not_working
After:    card_arrival
----------------------------------------------------------------------
Query:    I want to go back on what I did this morning
Expected: cancel_transfer
Before:   compromised_card


In [45]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/qlora-banking77"
)

PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(PROJECT_DIR)

/content/drive/MyDrive/qlora-banking77


In [46]:
RESULTS_DIR = PROJECT_DIR / "results"
ADAPTER_DIR = PROJECT_DIR / "adapter"
OUTPUT_DIR = PROJECT_DIR / "outputs"

for directory in [
    RESULTS_DIR,
    ADAPTER_DIR,
    OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

In [47]:
RESULTS_DIR = PROJECT_DIR / "results"

In [48]:
output_dir=str(
    PROJECT_DIR / "outputs" / "qlora-banking77"
)

In [49]:
ADAPTER_DIR = (
    PROJECT_DIR
    / "adapter"
    / "banking77-qwen3-1.7b-qlora"
)

ADAPTER_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [50]:
model.save_pretrained(
    ADAPTER_DIR
)

tokenizer.save_pretrained(
    ADAPTER_DIR
)

('/content/drive/MyDrive/qlora-banking77/adapter/banking77-qwen3-1.7b-qlora/tokenizer_config.json',
 '/content/drive/MyDrive/qlora-banking77/adapter/banking77-qwen3-1.7b-qlora/chat_template.jinja',
 '/content/drive/MyDrive/qlora-banking77/adapter/banking77-qwen3-1.7b-qlora/tokenizer.json')

In [51]:
baseline_path = (
    RESULTS_DIR
    / "baseline.json"
)

In [52]:
finetuned_path = (
    RESULTS_DIR
    / "finetuned.json"
)

In [53]:
!cp -r /content/qlora-banking77 \
    /content/drive/MyDrive/

cp: cannot stat '/content/qlora-banking77': No such file or directory


In [54]:
from google.colab import drive
from datetime import datetime
from pathlib import Path
import subprocess

# Mount Google Drive
drive.mount("/content/drive")

# Create a timestamped backup directory
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = Path(
    f"/content/drive/MyDrive/colab_backup_{timestamp}"
)
backup_dir.mkdir(parents=True, exist_ok=True)

# Copy everything in /content except Google Drive itself
subprocess.run(
    [
        "rsync",
        "-a",
        "--info=progress2",
        "--exclude=drive/",
        "/content/",
        str(backup_dir) + "/",
    ],
    check=True,
)

print(f"\nBackup complete:\n{backup_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Backup complete:
/content/drive/MyDrive/colab_backup_20261006_133615


In [55]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import shutil

# Mount Drive
drive.mount("/content/drive")

# Destination in Drive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = Path(
    f"/content/drive/MyDrive/qlora_backup_{timestamp}"
)
backup_dir.mkdir(parents=True, exist_ok=True)

# Files/directories that are useful for this project
patterns = [
    "baseline.json",
    "finetuned.json",
    "adapter_config.json",
    "*.safetensors",
    "*.bin",
    "trainer_state.json",
    "training_args.bin",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

found_files = []

# Search all /content, but do not recursively search mounted Drive
for pattern in patterns:
    for path in Path("/content").rglob(pattern):

        if str(path).startswith("/content/drive/"):
            continue

        if path.is_file():
            found_files.append(path)

# Remove duplicates
found_files = list(dict.fromkeys(found_files))

print("\nFOUND FILES")
print("=" * 70)

if not found_files:
    print("No saved model/result files were found in /content.")
else:
    for path in found_files:
        print(path)

        # Preserve a simple directory structure in Drive
        relative_path = path.relative_to("/content")
        destination = backup_dir / relative_path

        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            path,
            destination,
        )

print("\n" + "=" * 70)
print(f"Backup directory:\n{backup_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

FOUND FILES
/content/results/baseline.json
/content/results/finetuned.json
/content/adapter/banking77-qwen3-1.7b-qlora/adapter_config.json
/content/adapter/banking77-qwen3-1.7b-qlora/adapter_model.safetensors
/content/adapter/banking77-qwen3-1.7b-qlora/tokenizer.json
/content/adapter/banking77-qwen3-1.7b-qlora/tokenizer_config.json

Backup directory:
/content/drive/MyDrive/qlora_backup_20261006_133959


In [56]:
from pathlib import Path

ADAPTER_DIR = Path("adapter/banking77-qwen3-1.7b-qlora")

print("Adapter directory:", ADAPTER_DIR.resolve())

for path in sorted(ADAPTER_DIR.iterdir()):
    print(path.name)

Adapter directory: /content/adapter/banking77-qwen3-1.7b-qlora
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json


In [57]:
import gc
import torch

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

print(
    f"Allocated GPU memory: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

Allocated GPU memory: 1.89 GB


In [58]:
import gc
import torch

if "model" in globals():
    del model

if "trainer" in globals():
    del trainer

gc.collect()
torch.cuda.empty_cache()

In [59]:
from transformers import BitsAndBytesConfig

COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

print("Compute dtype:", COMPUTE_DTYPE)

Compute dtype: torch.bfloat16


In [60]:
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype="auto",
)

base_model.eval()

print("Fresh base model loaded.")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Fresh base model loaded.


In [61]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)

model.eval()

print("LoRA adapter loaded.")

LoRA adapter loaded.


In [62]:
print(model)
print("Active adapters:", model.active_adapters)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [63]:
trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters:     {total_params:,}")

Trainable parameters: 0
Total parameters:     1,033,364,480


In [64]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [65]:
query = "My card hasn't arrived yet."

print("Query:")
print(query)

print("\nPrediction:")
print(predict_intent(query))

Query:
My card hasn't arrived yet.

Prediction:
card_arrival


In [66]:
queries = [
    "How do I change my PIN?",
    "I don't recognize this cash withdrawal.",
    "My card isn't working.",
    "How can I activate my new card?",
    "I want to cancel a bank transfer.",
]

for query in queries:
    print("Query:     ", query)
    print("Prediction:", predict_intent(query))
    print("-" * 70)

Query:      How do I change my PIN?
Prediction: change_pin
----------------------------------------------------------------------
Query:      I don't recognize this cash withdrawal.
Prediction: cash_withdrawal_not_recognised
----------------------------------------------------------------------
Query:      My card isn't working.
Prediction: card_not_working
----------------------------------------------------------------------
Query:      How can I activate my new card?
Prediction: activate_my_card
----------------------------------------------------------------------
Query:      I want to cancel a bank transfer.
Prediction: cancel_transfer
----------------------------------------------------------------------


In [67]:
reloaded_results = evaluate_model(
    baseline_eval_ds
)

  0%|          | 0/100 [00:00<?, ?it/s]

In [68]:
print(
    f"Original fine-tuned accuracy: "
    f"{finetuned_results['accuracy']:.2%}"
)

print(
    f"Reloaded adapter accuracy:    "
    f"{reloaded_results['accuracy']:.2%}"
)

Original fine-tuned accuracy: 100.00%
Reloaded adapter accuracy:    99.00%


In [71]:
do_sample=False

In [72]:
prediction_matches = [
    original["predicted"] == reloaded["predicted"]
    for original, reloaded in zip(
        finetuned_results["predictions"],
        reloaded_results["predictions"],
    )
]

print(
    f"Identical predictions: "
    f"{sum(prediction_matches)}/"
    f"{len(prediction_matches)}"
)

Identical predictions: 99/100


In [73]:
for i, (original, reloaded) in enumerate(
    zip(
        finetuned_results["predictions"],
        reloaded_results["predictions"],
    )
):
    if original["predicted"] != reloaded["predicted"]:
        print("Index:    ", i)
        print("Query:    ", original["query"])
        print("Expected: ", original["expected"])
        print("Original: ", original["predicted"])
        print("Reloaded: ", reloaded["predicted"])

Index:     59
Query:     Can I freeze my card right now?
Expected:  compromised_card
Original:  compromised_card
Reloaded:  card_arrival


In [74]:
print(
    f"Before reload: "
    f"{finetuned_results['correct']}/"
    f"{finetuned_results['total']} "
    f"({finetuned_results['accuracy']:.2%})"
)

print(
    f"After reload:  "
    f"{reloaded_results['correct']}/"
    f"{reloaded_results['total']} "
    f"({reloaded_results['accuracy']:.2%})"
)

Before reload: 100/100 (100.00%)
After reload:  99/100 (99.00%)


In [75]:
lora_modules = [
    name
    for name, module in model.named_modules()
    if "lora_A" in name or "lora_B" in name
]

print("Number of LoRA modules:", len(lora_modules))

for name in lora_modules[:20]:
    print(name)

Number of LoRA modules: 784
base_model.model.model.layers.0.self_attn.q_proj.lora_A
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default
base_model.model.model.layers.0.self_attn.q_proj.lora_B
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default
base_model.model.model.layers.0.self_attn.k_proj.lora_A
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default
base_model.model.model.layers.0.self_attn.k_proj.lora_B
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default
base_model.model.model.layers.0.self_attn.v_proj.lora_A
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default
base_model.model.model.layers.0.self_attn.v_proj.lora_B
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default
base_model.model.model.layers.0.self_attn.o_proj.lora_A
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default
base_model.model.model.layers.0.self_attn.o_proj.lora_B
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default
base_model.m